# Various Wallet Selection

Same base data and copyable preselection as `stage1_wallet_strategy_selection` (strict,
stably-profitable wallets). Goal: find wallet coefficients `S(t) = sum_i alpha_i * copyable_pnl_i(t)`
with **minimal volatility at a targeted copyable PnL level**:

1. Load base trades + copyable preselection (same as stage1).
2. Group wallet trades by contract, normalize to a single token (net both-token buyers).
3. Similarity structure: per-contract vector cosine **and** daily-aggregated cosine/correlation.
4. Portfolio fit: minimize `Var(S)` s.t. `E[S]` over the train window equals a PnL target
   (default 500k), `alpha_i >= 0`, no `sum(alpha)=1` - the target pins exposure.
5. Adversarial validation: covariance shrinkage sensitivity, walk-forward CV with purge gap,
   **placebo permutation test**, out-of-sample splits, hourly plots.
6. Variant: per-(wallet, contract) notional caps - concentration, cap sweep, placebo per
   cap, sequential block folds.
7. Mechanism v2: bankroll-aware replication simulator - equity-following proportional
   copying with liquidity ceilings, capacity curve across book sizes, random-wallet placebo.
8. Wallet clusters: ward grouping on shrunk daily-pnl correlation, per-cluster performance
   curves, cluster-level vol-minimization re-test (placebo + WFCV) addressing the
   small-sample-covariance concern directly.

All portfolio analysis uses the **20m copyable PnL** (`copyable_pnl_20m_100`, per-fill
`(final_price - price) * copyable_qty_20m_100`) - a more conservative fill assumption;
preselection thresholds stay on stage1's base metrics.

### Headline findings (adversarial pass)
- In-sample vol reduction (~66% @ 500k) is reproduced almost exactly on **shuffled
  (mutually independent) wallet series** -> it is a parameter-count artifact, not diversification.
- Mean pairwise correlation of daily copyable pnl is only ~+0.08 (hourly +0.02): there is
  little persistent co-movement to harvest at any aggregation frequency.
- OOS ranking inverts vs in-sample (equal-weight best, minvar worst) - textbook noise-fitting.
- Universe copyable pnl collapses ~98% right after TRAIN_END (Jun->Jul 2026), so every
  OOS statistic lives in a broken regime.
- Per-contract notional caps (1k-5k) cut concentration and make the in-sample reduction
  statistically real (placebo ~100th pct), but forward block folds still fail and the
  capped equal-weight baseline loses its win tail OOS.
- Mechanism v2 (bankroll-mimicking replication) beats fixed-stake copying out-of-sample:
  at a $10k book it roughly doubles the bankroll in both test windows while fixed 1%/fill
  loses ~half; random-wallet placebos put the actual multiple above all placebo sets on
  the contract split - the selection edge survives a full trade-level simulation.
- Capacity is the binding constraint, not methodology: at a $100k book ~100% of fills hit
  the liquidity ceiling and per-window returns compress to +14%/+30%.
- Clustering partially vindicates the small-sample concern: at K=8 groups the in-sample
  vol reduction beats all shuffle placebos (100th pct vs 30th with 53 wallets), but it
  still does not persist forward (WFCV fails, shrinkage-insensitive) and cluster
  membership itself is unstable across train halves (ARI ~ 0).

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

## Configuration

In [2]:
MAX_LEAD_DAYS = 14  # keep only trades within this many days of contract resolution
TAGS = set(['Politics'])
TRAIN_START = pd.Timestamp("2025-09-01", tz="UTC")
TRAIN_END = pd.Timestamp("2026-07-01", tz="UTC")

TARGET_PNL = 500_000            # total copyable pnl targeted over the train period
FRONTIER_TARGETS = [250_000, 500_000, 1_000_000]
SHRINK_DELTAS = [0.0, 0.25, 0.5]
ALPHA_CAP = 50.0                # per-wallet coefficient cap (numerical sanity)
PLACEBO_N = 40                  # placebo permutation iterations
CAP_NOTIONAL = 2_000.0          # per-(wallet, contract) notional cap
CAPS_SWEEP = [1_000, 1_500, 2_000, 3_000, 5_000]
PLACEBO_N_CAP = 30              # placebo iterations per cap level

UTIL_TARGET = 0.5               # target mean concurrent deployment (fraction of bankroll)
B0_SIZES = [10_000, 100_000]    # reference book sizes for the capacity curve
PLACEBO_SIM_N = 12              # random-wallet placebo sets for the replication simulator

CLUSTER_KS = [4, 8]             # cluster counts for the wallet-grouping experiment
DELTA_SHRINK = 0.25             # correlation shrinkage before clustering

TRADES_DIR = Path("../../data/polygon_trades_processed")

## Load markets and trades (same base as stage1)

In [3]:
from polymarket_analysis.data.data_catalogue import load_markets_processed

mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & ((TAGS is None) or mdf["tags"].apply(lambda tags: any(tag in TAGS for tag in tags)))
    & (mdf['winner_token_id'].notna())
].copy().reset_index(drop=True)
print(f'selected markets for tags {TAGS}: {len(mdf):,}')

selected markets for tags {'Politics'}: 46,570


In [4]:
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat(
    [pd.read_parquet(f).merge(mdf, on="condition_id", how="inner") for f in trade_files],
    ignore_index=True,
)

df_full = df_full[df_full['side'] == 'BUY'].copy().reset_index(drop=True)
if TAGS is not None:
    df_full = df_full[df_full['primary_tag'].isin(TAGS)].copy().reset_index(drop=True)

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']
df_full['dt'] = pd.to_datetime(df_full['dt'], utc=True)

lead = (
    pd.to_datetime(df_full['last_condition_trade_ts'], utc=True, errors='coerce')
    - df_full['dt']
)
df_full = df_full[lead <= pd.Timedelta(days=MAX_LEAD_DAYS)].copy()
print(f'Lead filter (<= {MAX_LEAD_DAYS}d before resolution): {len(df_full):,} trades')

df_full = df_full.rename(columns={
    'total_quantity': 'quantity',
    'avg_price': 'price',
    'trade_value_usdc': 'usdc_amount',
})
for c in ['usdc_amount', 'final_value_usdc', 'quantity']:
    df_full[c] = df_full[c].astype(float)

# BUY-only stream
df_full['pnl'] = df_full['final_value_usdc'] - df_full['usdc_amount']
df_full['notional'] = df_full['usdc_amount']
df_full['is_train'] = df_full['last_condition_trade_ts'] <= TRAIN_END
print(f"train trades: {df_full['is_train'].sum():,}   test trades: {(~df_full['is_train']).sum():,}")

Lead filter (<= 14d before resolution): 6,715,807 trades


train trades: 5,905,079   test trades: 810,728


## Base + copyable preselection (same thresholds as stage1)

In [5]:
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics

wallet_vol_train, _ = compute_wallet_metrics(df_full[df_full['is_train']].copy())
wallet_vol_train['buy_copyable_roi'] = (
    wallet_vol_train['buy_copyable_pnl']
    / wallet_vol_train['buy_copyable_notional'].replace(0, np.nan)
)

candidates = wallet_vol_train.copy()
for col in ['average_roi', 'median_roi', 'num_buckets', 'num_markets',
            'pnl_volatility', 'max_drawdown_to_pnl',
            'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct',
            'market_pnl_hhi', 'copyable_roi', 'copyable_pnl_factor', 'copyable_pnl',
            'total_notional', 'max_copyable_drawdown_to_copyable_pnl', 'worst5_pnl_pct',
            'positive_bucket_share']:
    if col in candidates.columns:
        candidates[col] = pd.to_numeric(candidates[col], errors='coerce')

base_mask = (
    (candidates['total_pnl'] > 2000)
    & (candidates['buy_roi'] >= 0.1)
    & (candidates['copyable_pnl'] >= 1000)
    & (candidates['trade_count'] >= 300)
    & (candidates['num_buckets'] >= 30)
    & (candidates['estimated_buy_sharpe'] >= 3)
    & (candidates['market_pnl_hhi'].fillna(0.20) < 0.2)
    & (candidates['max_drawdown_to_pnl'] <= 0.2)
    & (candidates['median_dt'].dt.date <= (pd.Timestamp.today().date() - pd.Timedelta(days=30)))
    & (candidates['total_notional'] >= 5_000)
)
eligible_base = candidates[base_mask].copy()

copyable_mask = (
    (eligible_base['buy_copyable_pnl'] > 1000)
    & (eligible_base['buy_copyable_roi'] >= 0.05)
    & (eligible_base['estimated_copyable_buy_sharpe'] > 2.3)
)
copyable_group = (
    eligible_base[copyable_mask]
    .sort_values('copyable_pnl', ascending=False)
    .reset_index(drop=True)
)
copyable_wallets = copyable_group['wallet'].tolist()
N = len(copyable_wallets)

print(f'Base-eligible wallets: {len(eligible_base):,}')
print(f'Copyable group: {len(copyable_group):,} wallets, '
      f'total copyable pnl {copyable_group["copyable_pnl"].sum():,.0f}')
copyable_group[['wallet', 'copyable_pnl', 'buy_copyable_roi',
                'estimated_copyable_buy_sharpe']].head(20)

Base-eligible wallets: 72
Copyable group: 53 wallets, total copyable pnl 1,432,339


,wallet,copyable_pnl,buy_copyable_roi,estimated_copyable_buy_sharpe
0,0xfc2f4f50ce2f6045d35558a5e2d8d4b2ac6610c7,325189.98,0.41,2.54
1,0x9ad091ca2e8f1bd69f27662edcb49dceeaa5bf3d,152958.59,0.43,2.77
2,0x048215305cbcf7cc790735bf00119551d75c6b0a,102408.82,1.38,3.19
3,0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,77615.63,0.55,3.12
4,0x4da76bbf120899fc10fa6e0aad4bffdd19a7355e,70207.13,0.24,3.24
5,0x682c92615993fd1f75cdfe101efdc1a8adcb17ae,67907.76,0.47,2.55
6,0xd4140031e313f8d850740a80d2ee6653c925a4db,56837.42,0.21,3.38
7,0x641b56cd1de69da37e9bfefeddc7b277341c5433,53765.21,0.23,2.42
8,0xc0ff6a9ac424210cf218fda5c5753324c34a9953,51388.94,0.27,4.83
9,0xaf2383b66194112d57ac196b00ab25f78c17aa95,39229.77,0.17,3.66


## Contract normalization to a single token

Per contract the **reference token** is the token with the largest total traded quantity.
Fills of the opposite token are expressed as negative reference-token quantity
(buying the opposite token is a short position in the reference token), so wallets buying
both tokens of a contract are netted to one directional position.
Realized `pnl` / `copyable_pnl` are kept as-is and attributed to the net direction,
giving a 2-dim vector `[ref component, other component]` per wallet x contract.

In [6]:
ref_token = (
    df_full.groupby(['condition_id', 'token_id'], observed=True)['quantity'].sum()
    .reset_index()
    .sort_values('quantity')
    .drop_duplicates('condition_id', keep='last')
    .set_index('condition_id')['token_id']
)

d = df_full[df_full['wallet'].isin(set(copyable_wallets))].copy()
d['signed_qty'] = np.where(
    d['token_id'] == d['condition_id'].map(ref_token),
    d['quantity'], -d['quantity'],
)
# 20m copyable pnl variant (per-fill formula from stage1's COPYABLE_VARIANTS):
# more conservative fill assumption, more stable for politics
d['cpnl20'] = (d['final_price'] - d['price']) * d['copyable_qty_20m_100']
print(f"sum copyable_pnl(5m)={d['copyable_pnl'].sum():,.0f}  "
      f"sum cpnl20={d['cpnl20'].sum():,.0f}  "
      f"per-fill corr={d[['copyable_pnl', 'cpnl20']].corr().iloc[0, 1]:.3f}")

wc = d.groupby(['wallet', 'condition_id'], observed=True).agg(
    signed_qty=('signed_qty', 'sum'),
    pnl=('pnl', 'sum'),
    cpnl20=('cpnl20', 'sum'),
    notional=('notional', 'sum'),
).reset_index()
wc['dir_ref'] = wc['signed_qty'] >= 0
for src, ref_col, oth_col in [('pnl', 'v_pnl_ref', 'v_pnl_oth'),
                              ('cpnl20', 'v_cpn_ref', 'v_cpn_oth')]:
    wc[ref_col] = np.where(wc['dir_ref'], wc[src], 0.0)
    wc[oth_col] = np.where(wc['dir_ref'], 0.0, wc[src])

print(f'copyable-group fills: {len(d):,} across {d["condition_id"].nunique():,} contracts')
print(f'wallet-contract rows: {len(wc):,}, netted to other token: {(~wc["dir_ref"]).sum():,}')
wc.head()

sum copyable_pnl(5m)=1,958,435  sum cpnl20=2,697,715  per-fill corr=0.826
copyable-group fills: 169,201 across 5,749 contracts
wallet-contract rows: 11,672, netted to other token: 5,126


,wallet,condition_id,signed_qty,pnl,cpnl20,notional,dir_ref,v_pnl_ref,v_pnl_oth,v_cpn_ref,v_cpn_oth
0,0x01adea599a865a092b51ee4573581a75fc1390f6,0x0200743b501bb76457156e98736b1fd5f65835af8290...,-121.37,-47.46,-47.43,47.46,False,0.00,-47.46,0.00,-47.43
1,0x01adea599a865a092b51ee4573581a75fc1390f6,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720ef...,3241.09,-71.67,-71.67,71.67,True,-71.67,0.00,-71.67,0.00
2,0x01adea599a865a092b51ee4573581a75fc1390f6,0x0c8e90be724ba76a69c5d4dbf543a8afca2218b546d3...,-1467.20,-17.54,-17.54,17.54,False,0.00,-17.54,0.00,-17.54
3,0x01adea599a865a092b51ee4573581a75fc1390f6,0x1a716766423ff8a4e3f45b3cd69fad6c05d33452e29e...,493.42,-143.09,-110.63,143.09,True,-143.09,0.00,-110.63,0.00
4,0x01adea599a865a092b51ee4573581a75fc1390f6,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,-376.92,-107.45,-107.45,107.45,False,0.00,-107.45,0.00,-107.45


## Wallet similarity grid (per contract, vector cosine)

For each contract, wallets active in it are represented by their 2-dim
`[ref, other]` vector; pairwise cosine similarity is averaged over all contracts
two wallets share. Computed for wallet pnl and for copyable(20m) pnl.

In [7]:
wallet_idx = {w: i for i, w in enumerate(copyable_wallets)}

def similarity_grid(ref_col, oth_col):
    sim_sum = np.zeros((N, N))
    sim_cnt = np.zeros((N, N))
    for _, g in wc.groupby('condition_id', observed=True):
        if len(g) < 2:
            continue
        idx = g['wallet'].map(wallet_idx).to_numpy()
        V = g[[ref_col, oth_col]].to_numpy(dtype=float)
        nz = np.abs(V).sum(axis=1) > 0
        V, idx = V[nz], idx[nz]
        if len(V) < 2:
            continue
        norm = np.linalg.norm(V, axis=1)
        ok = norm > 0
        V, idx, norm = V[ok], idx[ok], norm[ok]
        if len(V) < 2:
            continue
        C = (V @ V.T) / np.outer(norm, norm)
        np.add.at(sim_sum, np.ix_(idx, idx), C)
        np.add.at(sim_cnt, np.ix_(idx, idx), 1.0)
    with np.errstate(divide='ignore', invalid='ignore'):
        sim = np.where(sim_cnt > 0, sim_sum / sim_cnt, np.nan)
    return sim, sim_cnt

sim_pnl, shared_cnt = similarity_grid('v_pnl_ref', 'v_pnl_oth')
sim_cpn, _ = similarity_grid('v_cpn_ref', 'v_cpn_oth')

off = ~np.eye(N, dtype=bool)
print(f'pairs sharing >=1 contract: {(shared_cnt[off] > 0).sum() // 2:,} of {N * (N - 1) // 2:,}')
print(f'median shared contracts per pair: {np.nanmedian(shared_cnt[off][shared_cnt[off] > 0]):.0f}')
print(f'sim (wallet pnl):    mean={np.nanmean(sim_pnl[off]):.3f}  '
      f'min={np.nanmin(sim_pnl[off]):.3f}  max={np.nanmax(sim_pnl[off]):.3f}')
print(f'sim (copyable 20m):  mean={np.nanmean(sim_cpn[off]):.3f}  '
      f'min={np.nanmin(sim_cpn[off]):.3f}  max={np.nanmax(sim_cpn[off]):.3f}')

pairs sharing >=1 contract: 1,114 of 1,378
median shared contracts per pair: 13
sim (wallet pnl):    mean=0.483  min=-1.000  max=1.000
sim (copyable 20m):  mean=0.414  min=-1.000  max=1.000


In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

short = [w[:8] for w in copyable_wallets]
fig = make_subplots(rows=1, cols=2, subplot_titles=('wallet pnl similarity', 'copyable pnl similarity'))
for col, sim in [(1, sim_pnl), (2, sim_cpn)]:
    fig.add_trace(go.Heatmap(
        z=sim, x=short, y=short, zmin=-1, zmax=1,
        colorscale='RdBu', colorbar=dict(x=0.45 * col),
    ), row=1, col=col)
fig.update_layout(width=1200, height=600, title='Per-contract wallet similarity (avg cosine)')
fig.show()

### Alternative aggregation: days instead of markets

The per-contract grid above measures *directional* agreement (same side of the same market).
The optimizer, however, exploits co-movement of **daily pnl series** - so also compute
cosine / Pearson correlation directly on daily copyable(20m) vectors (train window,
zero-filled days), and check how average pairwise correlation changes with the
aggregation frequency (hourly / daily / weekly).

In [9]:
d['hour'] = d['dt'].dt.floor('h')
d['day'] = d['dt'].dt.floor('D')
d['week'] = d['dt'].dt.floor('7D')
tr_active = d[d['is_train'] & (d['day'] >= TRAIN_START)]
FREQ = {'hour': 'h', 'day': 'D', 'week': '7D'}

def freq_matrix(col):
    g = (tr_active.groupby([col, 'wallet'], observed=True)['cpnl20']
         .sum().unstack('wallet').reindex(columns=copyable_wallets))
    full = pd.date_range(g.index.min(), g.index.max(), freq=FREQ[col])
    return g.reindex(full).fillna(0.0)

# daily cosine similarity grid (days instead of markets)
X_day = freq_matrix('day')
Ad = X_day.values
norms = np.linalg.norm(Ad, axis=0)
idx_ok = np.where(norms > 0)[0]
Ccos = (Ad[:, idx_ok].T @ Ad[:, idx_ok]) / np.outer(norms[idx_ok], norms[idx_ok])
sim_daily = np.full((N, N), np.nan)
sim_daily[np.ix_(idx_ok, idx_ok)] = Ccos
off_f = ~np.eye(N, dtype=bool)
vals = sim_daily[off_f]
print(f'daily-cos sim (train): mean={np.nanmean(vals):+.3f} median={np.nanmedian(vals):+.3f} '
      f'min={np.nanmin(vals):+.3f} max={np.nanmax(vals):+.3f}')
print(f'(per-contract grid: mean={np.nanmean(sim_cpn[off]):.3f})')

corr_rows = []
for label in ['hour', 'day', 'week']:
    Xf = freq_matrix(label)
    Cc = np.corrcoef(Xf.values.T)
    corr_rows.append({'agg': label, 'buckets': len(Xf),
                      'mean pairwise corr': float(np.nanmean(Cc[off_f])),
                      'share |corr|>0.2': float(np.nanmean(np.abs(Cc[off_f]) > 0.2))})
pd.DataFrame(corr_rows).set_index('agg')

daily-cos sim (train): mean=+0.090 median=+0.012 min=-0.275 max=+0.873
(per-contract grid: mean=0.414)


,buckets,mean pairwise corr,share |corr|>0.2
agg,,,
hour,7264,0.02,0.02
day,303,0.08,0.18
week,44,0.15,0.35


### Regime context: monthly universe copyable(20m) PnL

Before fitting, check how stable the underlying pnl stream is across months -
a portfolio target set on the train window is only meaningful if the regime persists.

In [10]:
monthly = d.groupby(d['dt'].dt.to_period('M'))['cpnl20'].sum()
print(monthly.to_string(float_format=lambda x: f'{x:,.0f}'))
print(f"train months total : {d[d['is_train']]['cpnl20'].sum():,.0f}")
print(f"post-TRAIN_END     : {d[~d['is_train']]['cpnl20'].sum():,.0f}")

dt
2025-01       1,285
2025-02         369
2025-03         -71
2025-04        -482
2025-05        -264
2025-06      40,355
2025-07      -1,725
2025-08         990
2025-09       8,505
2025-10      14,394
2025-11       3,193
2025-12       8,875
2026-01      63,129
2026-02     804,178
2026-03      74,193
2026-04     351,169
2026-05     -51,360
2026-06   1,348,144
2026-07      27,511
2026-08       5,327
Freq: M
train months total : 2,669,127
post-TRAIN_END     : 28,588


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_59010/3583942330.py:1: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



## Wallet coefficients: minimize volatility at a targeted copyable PnL

Combined series `S(t) = sum_i alpha_i * copyable_pnl_i(t)` on **daily, zero-filled** buckets
(same convention as `estimated_copyable_buy_sharpe`), restricted to active months
(`day >= TRAIN_START`). Fit on train:

$$\min_alpha \ alpha^T \Sigma_\delta \ alpha \quad \text{s.t.} \quad \mathbb{E}_{train}[S] = \text{TARGET\_PNL}, \ 0 \le alpha_i \le \text{ALPHA\_CAP}$$

with shrunk covariance $\Sigma_\delta = (1-\delta) \Sigma_{sample} + \delta \, \mathrm{diag}(\Sigma_{sample})$.
No `sum(alpha)=1` constraint: the pnl target pins overall scale/exposure.
Baselines: plain equal weights (`sum=1`), equal weights rescaled to the same target,
and **inverse-vol weights** (`delta=1`, i.e. diagonal covariance only - uses no
cross-wallet information).

In [11]:
d_fit = d[d['day'] >= TRAIN_START].copy()

def daily_matrix(sub):
    g = sub.groupby(['day', 'wallet'], observed=True)['cpnl20'].sum().unstack('wallet')
    g = g.reindex(columns=copyable_wallets)
    full = pd.date_range(g.index.min(), g.index.max(), freq='D')
    return g.reindex(full).fillna(0.0)

X_train = daily_matrix(d_fit[d_fit['is_train']])
X_test_c = daily_matrix(d_fit[~d_fit['is_train']])
X_test_t = daily_matrix(d_fit[d_fit['day'] > TRAIN_END])
print(f'train ({X_train.index.min().date()} .. {X_train.index.max().date()}): {X_train.shape} | '
      f'test contract-split: {X_test_c.shape} | test time-split: {X_test_t.shape}')

A_tr = X_train.values
T_tr = A_tr.shape[0]

def sample_cov(A):
    Ac = A - A.mean(axis=0)
    return Ac.T @ Ac / len(A)

def shrunk_cov(A, delta):
    S = sample_cov(A)
    if delta <= 0:
        return S
    return (1.0 - delta) * S + delta * np.diag(np.diag(S))

def minvar_at_target(A, target_total, delta=0.0, x0=None):
    """Min w'Sw s.t. expected total pnl over A equals target_total, long-only, capped.

    Solved in $k units for numerical conditioning."""
    n = A.shape[1]
    T = A.shape[0]
    Ask = A / 1e3
    tgt_k = target_total / 1e3
    mu = Ask.mean(axis=0)
    S = shrunk_cov(Ask, delta)

    def obj(w):
        return float(w @ S @ w)

    def grad(w):
        return 2.0 * (S @ w)

    cons = [{'type': 'eq',
             'fun': lambda w: float(mu @ w) * T - tgt_k,
             'jac': lambda w: mu * T}]
    if x0 is None:
        w_eq = np.full(n, 1.0 / n)
        x0 = w_eq * tgt_k / max(float(mu @ w_eq) * T, 1e-9)
    res = minimize(obj, x0, jac=grad, method='SLSQP',
                   bounds=[(0.0, ALPHA_CAP)] * n, constraints=cons,
                   options={'maxiter': 800, 'ftol': 1e-10})
    if not res.success:
        print(f'  [minvar warning] status={res.status}: {res.message}')
    return np.clip(res.x, 0.0, None)

w_eq = np.full(N, 1.0 / N)

def scaled_to_target(w, target):
    return w * target / max((A_tr @ w).sum(), 1e-9)

fits = {}
x_prev = None
for tgt in FRONTIER_TARGETS:
    w_mv = minvar_at_target(A_tr, tgt, delta=0.0, x0=x_prev)
    x_prev = w_mv.copy()
    fits[tgt] = {'minvar': w_mv, 'equal_scaled': scaled_to_target(w_eq, tgt)}

# diagonal-only covariance (delta=1): inverse-variance weights, no cross-wallet info
w_invvol = minvar_at_target(A_tr, TARGET_PNL, delta=1.0)
fits_delta = {dl: minvar_at_target(A_tr, TARGET_PNL, delta=dl) for dl in SHRINK_DELTAS}
print('frontier fitted (shrinkage delta=0); shrinkage fits at '
      f'{TARGET_PNL:,}: deltas {SHRINK_DELTAS} (+delta=1 inv-vol)')

train (2025-09-01 .. 2026-06-30): (303, 53) | test contract-split: (66, 53) | test time-split: (51, 53)


frontier fitted (shrinkage delta=0); shrinkage fits at 500,000: deltas [0.0, 0.25, 0.5] (+delta=1 inv-vol)


In [12]:
def eval_row(name, w):
    row = {'variant': name, 'sum(alpha)': float(w.sum()),
           'nonzero': int((w > 1e-4).sum())}
    for label, X in [('train', X_train), ('test_contract', X_test_c), ('test_time', X_test_t)]:
        s = X.values @ w
        row[f'{label} pnl'] = s.sum()
        row[f'{label} vol/d'] = s.std()
        row[f'{label} pnl/vol'] = s.mean() / s.std() if s.std() > 0 else np.nan
    return row

rows = [eval_row('equal-weight (sum=1)', w_eq)]
for tgt in FRONTIER_TARGETS:
    rows.append(eval_row(f'equal-scaled @{tgt/1000:.0f}k', fits[tgt]['equal_scaled']))
for tgt in FRONTIER_TARGETS:
    rows.append(eval_row(f'minvar @{tgt/1000:.0f}k (d=0)', fits[tgt]['minvar']))
rows.append(eval_row(f'inv-vol @500k (d=1)', w_invvol))
for dl in [dl for dl in SHRINK_DELTAS if dl != 0.0]:
    rows.append(eval_row(f'minvar @500k (d={dl})', fits_delta[dl]))

results = pd.DataFrame(rows)
cols = ['variant', 'sum(alpha)', 'nonzero'] + \
       [f'{sp} {m}' for sp in ['train', 'test_contract', 'test_time']
        for m in ['pnl', 'vol/d', 'pnl/vol']]
results[cols]

,variant,sum(alpha),nonzero,train pnl,train vol/d,train pnl/vol,test_contract pnl,test_contract vol/d,test_contract pnl/vol,test_time pnl,test_time vol/d,test_time pnl/vol
0,equal-weight (sum=1),1.00,53,49597.56,802.24,0.20,539.40,48.77,0.17,562.38,54.17,0.20
1,equal-scaled @250k,5.04,53,250000.00,4043.73,0.20,2718.90,245.82,0.17,2834.70,273.04,0.20
2,equal-scaled @500k,10.08,53,500000.00,8087.46,0.20,5437.80,491.64,0.17,5669.41,546.09,0.20
3,equal-scaled @1000k,20.16,53,1000000.00,16174.92,0.20,10875.59,983.28,0.17,11338.82,1092.18,0.20
4,minvar @250k (d=0),38.86,33,250000.00,1365.14,0.60,2473.02,411.27,0.09,2778.98,456.49,0.12
5,minvar @500k (d=0),77.71,33,500000.00,2730.29,0.60,4945.92,822.54,0.09,5557.94,912.97,0.12
6,minvar @1000k (d=0),155.43,33,1000000.00,5460.58,0.60,9891.84,1645.08,0.09,11115.89,1825.94,0.12
7,inv-vol @500k (d=1),63.63,53,500000.00,3982.31,0.41,1246.27,609.06,0.03,1471.39,681.50,0.04
8,minvar @500k (d=0.25),79.95,39,500000.00,2751.72,0.60,5483.58,846.41,0.10,6418.89,943.90,0.13
9,minvar @500k (d=0.5),79.82,49,500000.00,2849.13,0.58,4991.79,794.73,0.10,5998.25,886.37,0.13


### Stability: walk-forward CV inside train

Expanding-window monthly folds with a **7-day purge gap**: refit
`minvar @ pro-rated target` on all days up to `month_start - 7d`
(requiring >=100 post-purge history days), evaluate on the next month against the
equal-scaled baseline refit the same way. Summaries use median / IQR so single bad
months do not dominate.

In [13]:
idx = X_train.index
month_key = idx.strftime('%Y-%m')
fold_rows = []
for m in sorted(set(month_key)):
    pstart = pd.Period(m, 'M').start_time.tz_localize('UTC')
    te_m = month_key == m
    tr_m = (idx >= TRAIN_START) & (idx < pstart - pd.Timedelta(days=7))
    if te_m.sum() < 15 or tr_m.sum() < 100:
        continue
    Atr, Ate = A_tr[tr_m], A_tr[te_m]
    tgt_f = TARGET_PNL * tr_m.sum() / T_tr
    w_bl = w_eq * tgt_f / (Atr @ w_eq).sum()
    for dl in SHRINK_DELTAS:
        w_mv = minvar_at_target(Atr, tgt_f, delta=dl)
        r_mv, r_bl = Ate @ w_mv, Ate @ w_bl
        fold_rows.append({
            'month': m, 'delta': dl,
            'minvar vol/d': r_mv.std(), 'baseline vol/d': r_bl.std(),
            'vol reduction %': 100 * (1 - r_mv.std() / r_bl.std()),
            'minvar pnl/vol': r_mv.mean() / r_mv.std(),
            'baseline pnl/vol': r_bl.mean() / r_bl.std(),
        })

cv = pd.DataFrame(fold_rows)
summary = (cv.groupby('delta')['vol reduction %']
           .agg(median='median', q25=lambda s: s.quantile(.25),
                q75=lambda s: s.quantile(.75))
           )
wins = cv.assign(win=cv['vol reduction %'] > 0).groupby('delta')['win'].agg(['sum', 'count'])
summary['win_rate'] = wins['sum'] / wins['count']
print(cv.pivot(index='month', columns='delta', values='vol reduction %')
        .to_string(float_format=lambda x: f'{x:8.1f}'))
print('\nvol-reduction summary by delta:')
print(summary.to_string(float_format=lambda x: f'{x:8.1f}'))

best_delta = summary['median'].idxmax()
print(f'\nbest delta by median vol reduction: {best_delta}')

delta       0.00     0.25     0.50
month                             
2026-01   -251.3   -249.6   -247.8
2026-02    -50.4      6.3     16.2
2026-03     27.5     30.1     28.7
2026-04     27.1     40.4     46.6
2026-05   -117.0    -29.0   -169.1
2026-06     -8.4     -8.0     -7.7

vol-reduction summary by delta:
        median      q25      q75  win_rate
delta                                     
0.00     -29.4   -100.4     18.2       0.3
0.25      -0.8    -23.7     24.2       0.5
0.50       4.3   -128.8     25.6       0.5

best delta by median vol reduction: 0.5


### Adversarial check: placebo permutation test

Key question: is the in-sample vol reduction real diversification or an artifact of
optimizing 53 free parameters against a noisy covariance? **Placebo**: independently
shuffle each wallet's daily series within train - this destroys all cross-wallet
correlation while preserving each wallet's marginal distribution exactly. Refit minvar
on each shuffled matrix. If the "actual" in-sample reduction falls inside the placebo
distribution, it is pure noise-mining.

In [14]:
rng = np.random.default_rng(7)
w_bl_scaled = w_eq * TARGET_PNL / (A_tr @ w_eq).sum()

def in_sample_vol_red(Am):
    w = minvar_at_target(Am, TARGET_PNL, delta=0.0)
    return 100 * (1 - (Am @ w).std() / (Am @ w_bl_scaled).std())

vol_red_actual = in_sample_vol_red(A_tr)
print(f'ACTUAL in-sample vol reduction @500k: {vol_red_actual:.1f}%')

placebo = []
for b in range(PLACEBO_N):
    Ap = A_tr.copy()
    for j in range(N):
        Ap[:, j] = rng.permutation(Ap[:, j])
    placebo.append(in_sample_vol_red(Ap))
placebo = np.array(placebo)

pct = 100 * (placebo < vol_red_actual).mean()
print(f'PLACEBO ({PLACEBO_N} shuffles): mean={placebo.mean():.1f}%  '
      f'median={np.median(placebo):.1f}%  '
      f'p10-p90=[{np.percentile(placebo, 10):.1f}, {np.percentile(placebo, 90):.1f}]')
print(f'percentile of actual within placebo distribution: {pct:.0f}%')
if pct < 95:
    print('=> actual reduction NOT distinguishable from noise-mining on independent series')

fig = go.Figure(go.Histogram(x=placebo, nbinsx=12, name='placebo'))
fig.add_vline(x=vol_red_actual, line_width=3, line_color='red')
fig.add_annotation(x=vol_red_actual, yref='paper', y=1.02,
                   text='actual', showarrow=False, font_color='red')
fig.update_layout(title=f'In-sample vol reduction: placebo vs actual (%)',
                  xaxis_title='in-sample vol reduction %', width=800, height=400)
fig.show()

ACTUAL in-sample vol reduction @500k: 66.2%


PLACEBO (40 shuffles): mean=67.2%  median=66.9%  p10-p90=[65.6, 69.0]
percentile of actual within placebo distribution: 30%
=> actual reduction NOT distinguishable from noise-mining on independent series


### Fitted coefficients (minvar @ 500k, CV-best shrinkage)

In [15]:
w_main = fits_delta[best_delta]

weights = pd.DataFrame({
    'wallet': copyable_wallets,
    'alpha': w_main,
}).merge(
    copyable_group[['wallet', 'copyable_pnl', 'estimated_copyable_buy_sharpe']],
    on='wallet', how='left',
).sort_values('alpha', ascending=False).reset_index(drop=True)

eff_n = 1.0 / np.sum((w_main / w_main.sum()) ** 2) if w_main.sum() > 0 else 0.0
print(f'delta={best_delta}  sum(alpha)={w_main.sum():.2f}  nonzero={(w_main > 1e-4).sum()}  '
      f'effective N={eff_n:.1f}')
weights.head(25)

delta=0.5  sum(alpha)=79.82  nonzero=49  effective N=16.2


,wallet,alpha,copyable_pnl,estimated_copyable_buy_sharpe
0,0x6335b3ad24f6fd814c9bd083e4ab79926904ca2b,9.42,2015.19,11.49
1,0x7656ed7f597a0a61cd307591db198a42b2a7194b,8.20,3690.80,2.59
2,0xdf0850b029c97de282c491e91ce500384782cf97,7.13,1231.44,2.87
3,0xe8a75539bbb9041cd7b354eba400069cd8fc46c4,6.23,1770.86,3.83
4,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,5.43,3671.78,2.83
5,0xba68b89cb1ba9741cc1efcc8e2aae9fa6ce439d7,4.98,1197.94,5.48
6,0xac6b66982a6e7fec8a87eaf1baafc907feb7a962,4.55,3251.99,4.23
7,0x27b820e5203aa114acc2712e0e1d0ad758abb68c,4.24,2146.01,2.73
8,0x5bfa0627d533e29faeac9255cd8150a444d4f779,3.49,1044.14,2.61
9,0x9ffaa9cd1fb96e1fd635cae0a13e2022827fc8fc,3.35,2397.18,3.25


In [16]:
sel_idx = np.array([wallet_idx[w] for w in weights['wallet'][weights['alpha'] > 1e-4]])

def avg_pairwise(sim, idx):
    sub = sim[np.ix_(idx, idx)]
    m = ~np.eye(len(idx), dtype=bool)
    return np.nanmean(sub[m])

all_idx = np.arange(N)
print(f'avg pairwise sim, all wallets:     pnl={avg_pairwise(sim_pnl, all_idx):.3f}  '
      f'copyable20m={avg_pairwise(sim_cpn, all_idx):.3f}  daily-cos={avg_pairwise(sim_daily, all_idx):+.3f}')
print(f'avg pairwise sim, minvar-selected: pnl={avg_pairwise(sim_pnl, sel_idx):.3f}  '
      f'copyable20m={avg_pairwise(sim_cpn, sel_idx):.3f}  daily-cos={avg_pairwise(sim_daily, sel_idx):+.3f}')

avg pairwise sim, all wallets:     pnl=0.483  copyable20m=0.414  daily-cos=+0.090
avg pairwise sim, minvar-selected: pnl=0.483  copyable20m=0.412  daily-cos=+0.080


### Combined copyable PnL over time (hourly resolution)

In [17]:
d_fit['hour'] = d_fit['dt'].dt.floor('h')

def hourly_matrix(sub):
    g = sub.groupby(['hour', 'wallet'], observed=True)['cpnl20'].sum().unstack('wallet')
    g = g.reindex(columns=copyable_wallets)
    full = pd.date_range(g.index.min(), g.index.max(), freq='h')
    return g.reindex(full).fillna(0.0)

Xh_all = hourly_matrix(d_fit)
print(f'hourly matrix: {Xh_all.shape} ({Xh_all.index.min()} .. {Xh_all.index.max()})')

traces = [
    ('equal-weight (sum=1)', w_eq),
    ('equal-scaled @500k', fits[TARGET_PNL]['equal_scaled']),
    ('minvar @500k (d=0)', fits[TARGET_PNL]['minvar']),
    (f'minvar @500k (d={best_delta})', fits_delta[best_delta]),
    ('minvar @1000k (d=0)', fits[1_000_000]['minvar']),
]

fig = go.Figure()
for name, w in traces:
    s = pd.Series(Xh_all.values @ w, index=Xh_all.index).cumsum()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name))
fig.add_vline(x=TRAIN_END, line_dash='dash')
fig.add_annotation(x=TRAIN_END, yref='paper', y=1.02, text='TRAIN_END', showarrow=False)
fig.update_layout(title='Combined copyable PnL, hourly (cumulative)',
                  width=1100, height=500)
fig.show()

hourly matrix: (8498, 53) (2025-09-01 00:00:00+00:00 .. 2026-08-21 01:00:00+00:00)


In [18]:
Xh_test = Xh_all[Xh_all.index > TRAIN_END]

fig = go.Figure()
for name, w in traces:
    s = pd.Series(Xh_test.values @ w, index=Xh_test.index).cumsum()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name))
fig.update_layout(title='Combined copyable PnL, hourly (test window)',
                  width=1100, height=450)
fig.show()

test_stats = []
for name, w in traces:
    s = Xh_test.values @ w
    sd = X_test_c.values @ w
    test_stats.append({'variant': name,
                       'test hours pnl/vol': s.mean() / s.std(),
                       'daily pnl/vol': sd.mean() / sd.std()})
pd.DataFrame(test_stats)

,variant,test hours pnl/vol,daily pnl/vol
0,equal-weight (sum=1),0.05,0.17
1,equal-scaled @500k,0.05,0.17
2,minvar @500k (d=0),0.03,0.09
3,minvar @500k (d=0.5),0.03,0.10
4,minvar @1000k (d=0),0.03,0.09


## Per-contract notional cap (adversarial variant)

Question: does limiting copied notional per (wallet, contract) - e.g. 2k - improve
diversity and lower variance? Mechanism: fills are copied chronologically within each
wallet-contract until the cap is exhausted (boundary fill scaled). This truncates the
fat single-market spikes that dominate both variance and covariance estimates.
Checks below: concentration, cap sweep, **placebo test per cap** (is the in-sample
reduction real?), and sequential block folds (does it persist forward?).

In [19]:
d = d.sort_values(['wallet', 'condition_id', 'dt']).reset_index(drop=True)
prev_notional = (d.groupby(['wallet', 'condition_id'], observed=True)['notional']
                   .cumsum() - d['notional'])

def cap_factor(cap):
    return np.clip((cap - prev_notional) / d['notional'].clip(lower=1e-9), 0.0, 1.0)

def with_cap(cap):
    dd = d.copy()
    f = np.ones(len(dd)) if cap is None else cap_factor(cap)
    dd['cpnl20c'] = (dd['final_price'] - dd['price']) * dd['copyable_qty_20m_100'] * f
    return dd

def daily_matrix_col(sub, col='cpnl20c'):
    g = sub.groupby(['day', 'wallet'], observed=True)[col].sum().unstack('wallet')
    g = g.reindex(columns=copyable_wallets)
    full = pd.date_range(g.index.min(), g.index.max(), freq='D')
    return g.reindex(full).fillna(0.0)

sweep_rows = []
for cap in [None] + CAPS_SWEEP:
    label = 'uncapped' if cap is None else f'{cap/1000:g}k'
    sub = with_cap(cap)
    sub = sub[sub['day'] >= TRAIN_START]
    tr = sub[sub['is_train']]
    Xtr = daily_matrix_col(tr)
    Xtc = daily_matrix_col(sub[~sub['is_train']])
    Xtt = daily_matrix_col(sub[sub['day'] > TRAIN_END])
    A = Xtr.values
    train_total = A.sum()
    tgt = min(TARGET_PNL, 0.8 * train_total)

    wc_pnl = tr.groupby(['wallet', 'condition_id'], observed=True)['cpnl20c'].sum()
    w_pnl = tr.groupby('wallet')['cpnl20c'].sum()
    top5_wc = wc_pnl.nlargest(5).sum() / train_total
    top5_w = w_pnl.nlargest(5).sum() / train_total

    w_bl = w_eq * tgt / (A @ w_eq).sum()
    w_mv = minvar_at_target(A, tgt)
    row = {'cap': label, 'train total': train_total, 'target': tgt,
           'top5 wc share': top5_wc, 'top5 wallet share': top5_w,
           'mv sum(alpha)': w_mv.sum(), 'mv nonzero': int((w_mv > 1e-4).sum())}
    for nm, w in [('eq', w_bl), ('mv', w_mv)]:
        for tag, Xm in [('tr', A), ('tc', Xtc.values), ('tt', Xtt.values)]:
            s = Xm @ w
            row[f'{nm} {tag} pnl/vol'] = s.mean() / s.std()
        row[f'{nm} tr vol/d'] = (A @ w).std()
    sweep_rows.append(row)

sweep = pd.DataFrame(sweep_rows).set_index('cap')
sweep

,train total,target,top5 wc share,top5 wallet share,mv sum(alpha),mv nonzero,eq tr pnl/vol,eq tc pnl/vol,eq tt pnl/vol,eq tr vol/d,mv tr pnl/vol,mv tc pnl/vol,mv tt pnl/vol,mv tr vol/d
cap,,,,,,,,,,,,,,
uncapped,2628670.43,500000.00,0.56,0.58,77.71,33,0.20,0.17,0.20,8087.46,0.60,0.09,0.12,2730.29
1k,501787.03,401429.63,0.15,0.29,87.00,37,0.31,-0.02,0.04,4255.41,0.58,0.14,0.23,2276.73
1.5k,721574.48,500000.00,0.18,0.34,92.67,38,0.31,-0.04,0.01,5262.75,0.59,0.16,0.25,2803.14
2k,898141.70,500000.00,0.20,0.39,87.09,36,0.31,-0.04,-0.01,5352.21,0.60,0.14,0.23,2773.00
3k,1124605.57,500000.00,0.21,0.42,82.81,36,0.30,-0.05,-0.04,5443.01,0.60,0.04,0.07,2759.76
5k,1430186.92,500000.00,0.23,0.44,78.65,35,0.30,-0.02,-0.00,5573.53,0.60,0.04,0.07,2750.03


In [20]:
rng_cap = np.random.default_rng(7)
placebo_cap_rows = []
for cap in CAPS_SWEEP:
    sub = with_cap(cap)
    sub = sub[sub['day'] >= TRAIN_START]
    A = daily_matrix_col(sub[sub['is_train']]).values
    w_bl = w_eq * TARGET_PNL / (A @ w_eq).sum()

    def vol_red_c(Am):
        w = minvar_at_target(Am, TARGET_PNL)
        return 100 * (1 - (Am @ w).std() / (Am @ w_bl).std())

    actual = vol_red_c(A)
    pl = []
    for b in range(PLACEBO_N_CAP):
        Ap = A.copy()
        for j in range(N):
            Ap[:, j] = rng_cap.permutation(Ap[:, j])
        pl.append(vol_red_c(Ap))
    pl = np.array(pl)
    placebo_cap_rows.append({
        'cap': f'{cap/1000:g}k', 'actual %': actual, 'placebo mean %': pl.mean(),
        'placebo p90 %': np.percentile(pl, 90),
        'actual percentile': 100 * (pl < actual).mean()})
    print(f"cap={cap/1000:g}k: actual={actual:.1f}%  placebo mean={pl.mean():.1f}%  "
          f"p90={np.percentile(pl, 90):.1f}%  actual percentile={(pl < actual).mean()*100:.0f}%")

pd.DataFrame(placebo_cap_rows).set_index('cap')

cap=1k: actual=46.5%  placebo mean=36.4%  p90=39.3%  actual percentile=100%


cap=1.5k: actual=46.7%  placebo mean=37.6%  p90=39.9%  actual percentile=100%


cap=2k: actual=48.2%  placebo mean=39.2%  p90=42.2%  actual percentile=100%


cap=3k: actual=49.3%  placebo mean=41.3%  p90=44.1%  actual percentile=100%


cap=5k: actual=50.7%  placebo mean=45.1%  p90=48.2%  actual percentile=100%


,actual %,placebo mean %,placebo p90 %,actual percentile
cap,,,,
1k,46.50,36.43,39.28,100.00
1.5k,46.74,37.58,39.91,100.00
2k,48.19,39.21,42.15,100.00
3k,49.30,41.30,44.06,100.00
5k,50.66,45.07,48.19,100.00


In [21]:
sub2 = with_cap(CAP_NOTIONAL)
sub2 = sub2[sub2['day'] >= TRAIN_START]
X_all_c = daily_matrix_col(sub2)
idx_c = X_all_c.index
bounds = [pd.Timestamp(t, tz='UTC') for t in
          ['2025-12-01', '2026-03-01', '2026-06-01', '2026-07-01', '2026-09-01']]
fold_rows = []
for i in range(len(bounds) - 1):
    lo, hi = bounds[i], bounds[i + 1]
    fit_lo = TRAIN_START if i == 0 else bounds[i - 1]
    tr_mask = (idx_c >= fit_lo) & (idx_c < lo)
    te_mask = (idx_c >= lo) & (idx_c < hi)
    Atr, Ate = X_all_c.values[tr_mask], X_all_c.values[te_mask]
    if len(Atr) < 60 or len(Ate) < 15:
        print(f'fold {i}: skipped (tr={len(Atr)}, te={len(Ate)})')
        continue
    tgt_f = 0.5 * Atr.sum()   # leverage-matched target (~50% of fold history pnl)
    w_bl = w_eq * tgt_f / (Atr @ w_eq).sum()
    w_mv = minvar_at_target(Atr, tgt_f)
    r_mv, r_bl = Ate @ w_mv, Ate @ w_bl
    fold_rows.append({
        'fold': f'{fit_lo.date()}..{lo.date()} -> {hi.date()}',
        'vol reduction %': 100 * (1 - r_mv.std() / r_bl.std()),
        'mv pnl/vol': r_mv.mean() / r_mv.std(),
        'bl pnl/vol': r_bl.mean() / r_bl.std()})
pd.DataFrame(fold_rows).set_index('fold')

fold 3: skipped (tr=30, te=52)


,vol reduction %,mv pnl/vol,bl pnl/vol
fold,,,
2025-09-01..2025-12-01 -> 2026-03-01,-537.26,0.17,0.34
2025-12-01..2026-03-01 -> 2026-06-01,-149.10,0.30,0.30
2026-03-01..2026-06-01 -> 2026-07-01,-211.54,0.32,0.68


### Cap findings

- **Diversity: substantially improved.** Top-5 wallet-contract share of train pnl drops
  55% -> ~20% at the 2k cap; top-5 wallet share 58% -> 39%.
- **The in-sample vol reduction becomes *real*.** Placebo test per cap: the actual
  reduction beats ~100% of shuffled (independent-series) placebos at every cap 1k-5k,
  versus the 30th percentile uncapped. Removing single-market spikes exposes genuine
  (if modest) co-movement the optimizer can exploit; the excess over placebo (~7-8pp)
  is stable across the cap choice, so it is not a lucky cap.
- **But forward validation still fails.** Sequential block folds (leverage-matched
  targets) show negative vol reduction and minvar pnl/vol <= equal-weight in every fold.
  The capped equal-weight baseline also goes negative out-of-sample: capping truncates
  exactly the win tail that produced this universe's pnl (top-5 contracts were 55% of it).
- **Leverage explodes**: sum(alpha) ~ 80-110 at tight caps vs ~10 uncapped, because the
  capped universe's total pnl is much smaller than the 500k target.

Verdict: cap per-contract notional for risk/concentration reasons (it genuinely lowers
stream variance and single-market exposure), but combine the capped streams with equal or
inverse-vol weights - the minvar combination does not persist forward even with a cleaner
covariance.

## Wallet clusters and the small-sample covariance question

The minvar failure could be a dimensionality problem: 53 streams give 1,378 free
covariance parameters against 303 daily observations. If wallets form a few coherent
groups, aggregating within groups first collapses the estimation problem to K x K.
Test: ward clustering on `1 - shrunk corr` of daily cpnl20 (train only), then re-run the
portfolio comparison at cluster level - with placebo and walk-forward validation.

In [22]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score, silhouette_score

A_cl = X_train.values
T_cl, n_cl = A_cl.shape

def clust_dist(Asub):
    C = np.nan_to_num(np.corrcoef(Asub.T), nan=0.0)
    np.fill_diagonal(C, 1.0)
    Cs = (1 - DELTA_SHRINK) * C + DELTA_SHRINK * np.eye(Asub.shape[1])
    Dm = 1 - Cs
    Dm = (Dm + Dm.T) / 2
    np.fill_diagonal(Dm, 0.0)
    return Dm

def cluster_labels(Asub, k):
    Z = linkage(squareform(clust_dist(Asub), checks=False), method='ward')
    return Z, fcluster(Z, k, criterion='maxclust')

Z_full, _ = cluster_labels(A_cl, 2)

print('silhouette by k:')
for k in range(2, 9):
    lab = fcluster(Z_full, k, criterion='maxclust')
    sil = silhouette_score(clust_dist(A_cl), lab, metric='precomputed')
    print(f'  k={k}: sil={sil:+.3f}  sizes={sorted(np.bincount(lab)[1:].tolist(), reverse=True)}')

half = T_cl // 2
for k in CLUSTER_KS:
    l1 = cluster_labels(A_cl[:half], k)[1]
    l2 = cluster_labels(A_cl[half:], k)[1]
    print(f'split-half ARI k={k}: {adjusted_rand_score(l1, l2):+.3f}')

silhouette by k:
  k=2: sil=+0.083  sizes=[28, 25]
  k=3: sil=+0.073  sizes=[25, 17, 11]
  k=4: sil=+0.094  sizes=[19, 17, 11, 6]
  k=5: sil=+0.090  sizes=[19, 11, 11, 6, 6]
  k=6: sil=+0.104  sizes=[11, 11, 10, 9, 6, 6]
  k=7: sil=+0.115  sizes=[11, 10, 9, 6, 6, 6, 5]
  k=8: sil=+0.128  sizes=[11, 10, 6, 6, 6, 5, 5, 4]
split-half ARI k=4: -0.012
split-half ARI k=8: +0.015


/Users/vobornij/projects/polymarket/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3063: RuntimeWarning:

invalid value encountered in divide

/Users/vobornij/projects/polymarket/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3064: RuntimeWarning:

invalid value encountered in divide

/Users/vobornij/projects/polymarket/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3063: RuntimeWarning:

invalid value encountered in divide

/Users/vobornij/projects/polymarket/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3064: RuntimeWarning:

invalid value encountered in divide



### Cluster composition and per-cluster performance

In [23]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

Xf_cl = daily_matrix(d[d['day'] >= TRAIN_START])
C_raw = np.nan_to_num(np.corrcoef(Xf_cl[Xf_cl.index <= TRAIN_END].T), nan=0.0)
np.fill_diagonal(C_raw, 1.0)

CLUSTER_MEMBERS = {}
fig = make_subplots(rows=1, cols=len(CLUSTER_KS),
                    subplot_titles=[f'K={k} cumulative cpnl20 ($k)' for k in CLUSTER_KS])
for col_i, K in enumerate(CLUSTER_KS):
    labels = fcluster(Z_full, K, criterion='maxclust')
    members = {c: [copyable_wallets[j] for j in range(n_cl) if labels[j] == c]
               for c in range(1, K + 1)}
    CLUSTER_MEMBERS[K] = members
    print(f'K={K} cluster summary:')
    rows = []
    for c in range(1, K + 1):
        idx_c = [j for j in range(n_cl) if labels[j] == c]
        off = ~np.eye(len(idx_c), dtype=bool)
        other = [j for j in range(n_cl) if labels[j] != c]
        rows.append({
            'cluster': c, 'n': len(idx_c),
            'train pnl': Xf_cl.loc[Xf_cl.index <= TRAIN_END, members[c]].values.sum(),
            'test pnl': Xf_cl.loc[Xf_cl.index > TRAIN_END, members[c]].values.sum(),
            'within corr': C_raw[np.ix_(idx_c, idx_c)][off].mean(),
            'between corr': C_raw[np.ix_(idx_c, other)].mean()})
    print(pd.DataFrame(rows).set_index('cluster').round(3).to_string(), '\n')
    for c in range(1, K + 1):
        s = Xf_cl[members[c]].sum(axis=1).cumsum() / 1000
        fig.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines',
                                 name=f'K{K} c{c} (n={len(members[c])})',
                                 legendgroup=f'K{K}c{c}',
                                 showlegend=True), row=1, col=col_i + 1)
for ax_name in ['x', 'x2'][:len(CLUSTER_KS)]:
    fig.add_shape(type='line', x0=str(TRAIN_END.date()), x1=str(TRAIN_END.date()),
                  y0=0, y1=1, yref='paper', xref=ax_name,
                  line=dict(color='red', dash='dash'))
fig.update_layout(width=1200, height=450, title='per-cluster cumulative copyable pnl')
fig.show()

Xte_cl = Xf_cl[Xf_cl.index > TRAIN_END]
fig = make_subplots(rows=1, cols=len(CLUSTER_KS),
                    subplot_titles=[f'K={k} test-period cpnl20 ($k)' for k in CLUSTER_KS])
for col_i, K in enumerate(CLUSTER_KS):
    members = CLUSTER_MEMBERS[K]
    tot = Xte_cl.sum(axis=1).cumsum() / 1000
    fig.add_trace(go.Scatter(x=tot.index, y=tot.values, mode='lines',
                             name='all 53 wallets', line=dict(color='black', dash='dot'),
                             legendgroup='tot', showlegend=True),
                  row=1, col=col_i + 1)
    for c in range(1, K + 1):
        s = Xte_cl[members[c]].sum(axis=1).cumsum() / 1000
        fig.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines',
                                 name=f'K{K} c{c} (n={len(members[c])})',
                                 legendgroup=f'K{K}c{c}',
                                 showlegend=True), row=1, col=col_i + 1)
fig.update_layout(width=1200, height=450,
                  title=f'per-cluster cumulative copyable pnl - test period only '
                        f'(after {TRAIN_END.date()}; black dotted = all wallets)')
fig.show()

print('test-period totals per cluster ($):')
for K in CLUSTER_KS:
    print(f'K={K}: ' + '  '.join(
        f"c{c}={Xte_cl[CLUSTER_MEMBERS[K][c]].values.sum():+,.0f}" for c in range(1, K + 1)))

K=4 cluster summary:
          n  train pnl  test pnl  within corr  between corr
cluster                                                    
1         6  169660.09   1222.94         0.31          0.00
2        19  328602.74  11243.82         0.05          0.02
3        11 1346583.12  13235.50         0.38          0.06
4        17  782606.81   4103.75         0.24          0.06 

K=8 cluster summary:
          n  train pnl  test pnl  within corr  between corr
cluster                                                    
1         6  169660.09   1222.94         0.31          0.00
2        10   39565.51   5547.94         0.03         -0.01
3         4  240484.30    -96.42         0.40          0.05
4         5   48552.93   5792.30         0.31          0.05
5        11 1346583.12  13235.50         0.38          0.06
6         6  263417.78    230.14         0.64          0.10
7         5  164700.18  10416.18         0.31          0.07
8         6  354488.85  -6542.57         0.31          0

test-period totals per cluster ($):
K=4: c1=+1,223  c2=+11,244  c3=+13,236  c4=+4,104
K=8: c1=+1,223  c2=+5,548  c3=-96  c4=+5,792  c5=+13,236  c6=+230  c7=+10,416  c8=-6,543


### Cluster-level vol minimization: does shrinking the problem fix it?

Fit minvar across the K cluster aggregates (equal-mean within cluster) at the same total
pnl target. Compare OOS vol reduction vs equal-weight-across-clusters, the shuffle
placebo percentile, and monthly walk-forward folds - at several shrinkage levels.

In [24]:
from scipy.optimize import minimize

TARGET_CL = 500_000


def sample_cov(B):
    Bc = B - B.mean(axis=0)
    return Bc.T @ Bc / len(B)


def minvar_at_target(B, target_total, delta=0.0):
    m, Tb = B.shape[1], B.shape[0]
    Bsk, tgt_k = B / 1e3, target_total / 1e3
    mu = Bsk.mean(axis=0)
    S = sample_cov(Bsk)
    if delta > 0:
        S = (1 - delta) * S + delta * np.diag(np.diag(S))
    cons = [{'type': 'eq', 'fun': lambda w: float(mu @ w) * Tb - tgt_k,
             'jac': lambda w: mu * Tb}]
    x0 = np.full(m, tgt_k / m / max(float(mu.mean()), 1e-9))
    res = minimize(lambda w: float(w @ S @ w), x0, jac=lambda w: 2.0 * (S @ w),
                   method='SLSQP', bounds=[(0.0, 500.0)] * m, constraints=cons,
                   options={'maxiter': 800, 'ftol': 1e-10})
    return np.clip(res.x, 0.0, None)


cl_results = []
for K in CLUSTER_KS:
    members = CLUSTER_MEMBERS[K]
    Mmap = np.zeros((n_cl, K))
    for j in range(n_cl):
        for c in range(1, K + 1):
            if copyable_wallets[j] in members[c]:
                Mmap[j, c - 1] = 1.0
    Mmap = Mmap / Mmap.sum(axis=0, keepdims=True)

    Y_tr = A_cl @ Mmap
    Y_tc = X_test_c.values @ Mmap
    Y_tt = X_test_t.values @ Mmap
    Ky = K

    w_bl = np.full(Ky, 1.0 / Ky)
    w_bl = w_bl * TARGET_CL / (Y_tr @ w_bl).sum()

    # placebo + in-sample reduction per delta (computed once)
    rng_pl = np.random.default_rng(11)
    plc = {dl: [] for dl in [0.0, 0.25, 0.5]}
    for b in range(PLACEBO_N // 2):
        Yp = Y_tr.copy()
        for j in range(Ky):
            Yp[:, j] = rng_pl.permutation(Yp[:, j])
        wb_p = np.full(Ky, 1.0 / Ky)
        wb_p = wb_p * TARGET_CL / (Yp @ wb_p).sum()
        for dl in plc:
            wp = minvar_at_target(Yp, TARGET_CL, delta=dl)
            plc[dl].append(100 * (1 - (Yp @ wp).std() / (Yp @ wb_p).std()))
    is_red = {dl: 100 * (1 - (Y_tr @ minvar_at_target(Y_tr, TARGET_CL, delta=dl)).std()
                         / (Y_tr @ w_bl).std()) for dl in plc}
    pct = {dl: float((np.array(plc[dl]) < is_red[dl]).mean()) for dl in plc}

    for name, Yte in [('test_c', Y_tc), ('test_t', Y_tt)]:
        r_bl = Yte @ w_bl
        for dl in plc:
            wm = minvar_at_target(Y_tr, TARGET_CL, delta=dl)
            r_mv = Yte @ wm
            cl_results.append({'K': K, 'delta': dl, 'split': name,
                               'bl pnl/vol': r_bl.mean() / r_bl.std(),
                               'mv pnl/vol': r_mv.mean() / r_mv.std(),
                               'OOS vol red %': 100 * (1 - r_mv.std() / r_bl.std()),
                               'IS red %': is_red[dl],
                               'placebo pct': pct[dl]})
res_cl = pd.DataFrame(cl_results)
print(res_cl.set_index(['K', 'delta', 'split']).to_string(float_format=lambda x: f'{x:8.2f}'))

# WFCV medians per (K, delta)
fold_summary = []
idx_cl = X_train.index
mk = idx_cl.strftime('%Y-%m')
for K in CLUSTER_KS:
    members = CLUSTER_MEMBERS[K]
    Mmap = np.zeros((n_cl, K))
    for j in range(n_cl):
        for c in range(1, K + 1):
            if copyable_wallets[j] in members[c]:
                Mmap[j, c - 1] = 1.0
    Mmap = Mmap / Mmap.sum(axis=0, keepdims=True)
    Y_tr = A_cl @ Mmap
    w_bl = np.full(K, 1.0 / K)
    w_bl = w_bl * TARGET_CL / (Y_tr @ w_bl).sum()
    reds = {dl: [] for dl in [0.0, 0.5]}
    for m in sorted(set(mk)):
        pstart = pd.Period(m, 'M').start_time.tz_localize('UTC')
        te = mk == m
        tr_mask = (idx_cl >= TRAIN_START) & (idx_cl < pstart - pd.Timedelta(days=7))
        if te.sum() < 15 or tr_mask.sum() < 100:
            continue
        Btr, Bte = Y_tr[tr_mask], Y_tr[te]
        tgt_f = TARGET_CL * tr_mask.sum() / T_cl
        r_b = Bte @ (w_bl * tgt_f / TARGET_CL)
        for dl in reds:
            wm = minvar_at_target(Btr, tgt_f, delta=dl)
            r_m = Bte @ wm
            reds[dl].append(100 * (1 - r_m.std() / r_b.std()))
    for dl, rr in reds.items():
        fold_summary.append({'K': K, 'delta': dl,
                             'WFCV median red %': float(np.median(rr))})
print('\nwalk-forward folds:')
print(pd.DataFrame(fold_summary).set_index(['K', 'delta']).to_string(float_format=lambda x: f'{x:8.2f}'))

                bl pnl/vol  mv pnl/vol  OOS vol red %  IS red %  placebo pct
K delta split                                                               
4 0.00  test_c        0.19        0.19         -31.50     29.02         0.20
  0.25  test_c        0.19        0.19         -29.48     28.85         0.20
  0.50  test_c        0.19        0.19         -29.55     28.33         0.20
  0.00  test_t        0.22        0.23         -30.07     29.02         0.20
  0.25  test_t        0.22        0.23         -27.99     28.85         0.20
  0.50  test_t        0.22        0.23         -27.97     28.33         0.20
8 0.00  test_c        0.14        0.19        -386.24     48.61         1.00
  0.25  test_c        0.14        0.20        -370.66     48.42         1.00
  0.50  test_c        0.14        0.19        -352.01     47.71         1.00
  0.00  test_t        0.17        0.23        -395.16     48.61         1.00
  0.25  test_t        0.17        0.23        -379.09     48.42         1.00

### Cluster findings

- **Cluster structure is weak and unstable.** Silhouettes stay below 0.15 and rise
  monotonically with k; split-half ARI is ~ 0 (clusters do not reproduce between train
  halves). There are a few genuinely tight pockets (within-corr up to 0.64) but no clean
  partition into a few stable groups.
- **The small-sample concern is partially right**: with 53 wallets the in-sample vol
  reduction sat at the ~30th placebo percentile (indistinguishable from noise); aggregated
  to K=8 clusters it jumps above ALL shuffle placebos (~100th pct) - real cross-wallet
  co-movement exists but was buried under estimation noise at full dimensionality.
- **But detecting the signal is not exploiting it**: cluster-level OOS vol reduction is
  strongly negative at every K and shrinkage level (minvar *increases* OOS vol vs equal
  weight), and WFCV fold medians fail everywhere. The failure is regime rotation, not
  estimation error - shrinkage barely moves anything.
- **Performance is highly concentrated across groups**: one or two clusters carry most of
  the universe pnl (the tightest pocket earns ~$1.35M of $2.6M train), while the loose
  bulk contributes little and even loses money post-TRAIN_END.

Verdict: aggregation makes the covariance signal *measurable* but not *persistent*;
cluster-level combination does not rescue minvar.

## Bankroll-aware replication simulator (mechanism v2)

Instead of combining realized pnl streams (`alpha_i * copyable_pnl_i(t)`), simulate an
actual copy book over the selected wallets, fill by fill:

- **Equity-following sizing**: per fill stake `theta_w * frac_f * E(t)` where
  `frac_f = notional_f / B_hat_w` mimics the wallet's own fraction-of-bankroll bet
  (scale, don't clip - the wallet's conviction profile is the signal);
- **Bankroll proxy** `B_hat_w` = p75 per-contract notional over train only (most stable
  candidate across train halves; pnl high-water marks are unstable, rho=0.36);
- **Limits**: liquidity ceiling `copyable_qty_20m_100 * price` per fill and the
  per-(wallet, contract) notional cap;
- **Payouts at resolution**, cash accounting (open positions held at cost, no
  mark-to-market - a simplification vs reality);
- **Utilization normalization**: weights rescaled so a pure-mimicry pass (limits off)
  deploys on average `UTIL_TARGET` of the bankroll concurrently.

Baselines: naive fixed staking (1% of B0 per fill) and inverse-vol weights. Splits:
train / contract-split OOS / time-split OOS.

In [25]:
import heapq

d['res_ts'] = pd.to_datetime(d['last_condition_trade_ts'], utc=True)

tr_sim = d[d['is_train'] & (d['day'] >= TRAIN_START)]
bhat = (tr_sim.groupby(['wallet', 'condition_id'], observed=True)['notional'].sum()
              .groupby(level=0).quantile(0.75))

theta_ivol = 1.0 / X_train.std()
theta_ivol = theta_ivol / theta_ivol.mean()

SIM_SPLITS = {
    'train': tr_sim,
    'test_c': d[(~d['is_train']) & (d['day'] >= TRAIN_START)],
    'test_t': d[d['day'] > TRAIN_END],
}


def simulate(sub, bhat, theta_map=None, B0=10_000.0, cap_abs=CAP_NOTIONAL,
             util_target=UTIL_TARGET, enforce_limits=True, fixed_frac=None,
             return_eq=False):
    """Equity-following copy book, cash accounting."""
    f = sub.sort_values('dt', kind='mergesort')
    frac = (f['notional'] / f['wallet'].map(bhat)).to_numpy()
    th = (f['wallet'].map(theta_map if theta_map is not None else
                          dict.fromkeys(bhat.index, 1.0)).fillna(1.0).to_numpy())
    ceil_usd = (f['copyable_qty_20m_100'] * f['price']).to_numpy()
    price = f['price'].to_numpy()
    res_ts = f['res_ts'].to_numpy()
    fin_price = f['final_price'].to_numpy()
    dt = f['dt'].to_numpy()
    w_arr = f['wallet'].to_numpy()
    c_arr = f['condition_id'].to_numpy()

    mark_ts, mark_val, util_marks = [], [], []

    def core(theta_arr, ff):
        cash, open_usd = float(B0), 0.0
        cap_used, payouts = {}, []
        stats = {'n': len(f), 'traded': 0, 'cap_bind': 0, 'liq_bind': 0}

        def mark(t):
            mark_ts.append(t)
            mark_val.append(cash + open_usd)
            den = cash + open_usd
            util_marks.append(open_usd / den if den > 1e-9 else 0.0)

        def drain(until):
            nonlocal cash, open_usd
            while payouts and payouts[0][0] <= until:
                _, qty, fp, st = heapq.heappop(payouts)
                cash += qty * fp
                open_usd -= st
                mark(until)

        mark_ts.clear()
        mark_val.clear()
        util_marks.clear()
        mark(dt[0])
        for i in range(len(f)):
            drain(dt[i])
            desired = ff * B0 if ff is not None else theta_arr[i] * frac[i] * cash
            lim_liq = ceil_usd[i] if enforce_limits else np.inf
            key = (w_arr[i], c_arr[i])
            remaining = (np.inf if (cap_abs is None or not enforce_limits)
                         else cap_abs - cap_used.get(key, 0.0))
            stake = min(desired, lim_liq, remaining)
            if enforce_limits and lim_liq < desired:
                stats['liq_bind'] += 1
            if enforce_limits and remaining < min(desired, lim_liq):
                stats['cap_bind'] += 1
            if stake >= 1.0 and cash > 0:
                stake = min(stake, cash)
                qty = stake / price[i]
                cash -= stake
                open_usd += stake
                stats['traded'] += 1
                if enforce_limits and cap_abs is not None:
                    cap_used[key] = cap_used.get(key, 0.0) + stake
                heapq.heappush(payouts, (res_ts[i], qty, fin_price[i], stake))
                mark(dt[i])
        drain(dt[-1] + np.timedelta64(30, 'D'))
        eq = pd.Series(mark_val, index=pd.DatetimeIndex(mark_ts, tz='UTC'))
        eq = eq.groupby(level=0).last().sort_index()
        return eq.resample('D').last().ffill(), stats

    if util_target is not None:
        # pure-mimicry pass (limits off) to measure deployment demand
        cash, open_usd = float(B0), 0.0
        cap_used, payouts = {}, []
        us = []
        for i in range(len(f)):
            t = dt[i]
            while payouts and payouts[0][0] <= t:
                _, qty, fp, st = heapq.heappop(payouts)
                cash += qty * fp
                open_usd -= st
            desired = fixed_frac * B0 if fixed_frac is not None else th[i] * frac[i] * cash
            if desired >= 1.0 and cash > 0:
                stake = min(desired, cash)
                qty = stake / price[i]
                cash -= stake
                open_usd += stake
                heapq.heappush(payouts, (res_ts[i], qty, fin_price[i], stake))
            den = cash + open_usd
            us.append(open_usd / den if den > 1e-9 else 0.0)
        scale = util_target / max(float(np.mean(us)), 1e-9)
        th = th * scale
        if fixed_frac is not None:
            fixed_frac = fixed_frac * scale

    eq_d, stats = core(th, fixed_frac)
    ret = eq_d.pct_change().dropna().replace([np.inf, -np.inf], np.nan).dropna()
    sharpe = (ret.mean() / ret.std() * np.sqrt(365)
              if len(ret) > 5 and ret.std() > 0 else np.nan)
    dd = float((eq_d / eq_d.cummax() - 1).min()) if len(eq_d) > 1 else np.nan
    out = {'multiple': eq_d.iloc[-1] / B0, 'ann_sharpe': sharpe, 'maxDD': dd,
           'util%': 100 * float(np.mean(util_marks)),
           'liq_bind%': 100 * stats['liq_bind'] / stats['n']}
    if return_eq:
        out['eq'] = eq_d
    return out

### Variant grid across splits and book sizes

In [26]:
rows = []
for split_name, sub in SIM_SPLITS.items():
    for B0 in B0_SIZES:
        rows.append({'variant': 'fixed 1%/fill', 'B0': B0, 'cap': CAP_NOTIONAL,
                     **simulate(sub, bhat, B0=B0, fixed_frac=0.01)})
        for th_name in ['equal', 'ivol']:
            th_map = None if th_name == 'equal' else theta_ivol.to_dict()
            for cap in [CAP_NOTIONAL, None]:
                rows.append({'variant': th_name, 'B0': B0, 'cap': 'off' if cap is None else cap,
                             **simulate(sub, bhat, theta_map=th_map, B0=B0, cap_abs=cap)})
sim_grid = pd.DataFrame(rows).set_index(['variant', 'B0', 'cap'])
sim_grid

multiple  ann_sharpe  maxDD  util%  liq_bind%
variant       B0     cap                                                   
fixed 1%/fill 10000  2000.00    101.79        3.04  -0.20  24.20      95.90
equal         10000  2000.00    114.19        3.02  -0.19  25.78      96.87
                      off       192.69        3.01  -0.18  44.00      90.19
ivol          10000  2000.00    100.54        3.10  -0.09  24.60      85.08
                      off       163.79        3.17  -0.09  32.33      84.11
fixed 1%/fill 100000 2000.00     12.82        2.92  -0.06  15.26      99.72
equal         100000 2000.00     12.82        2.92  -0.06  15.26     100.00
                      off        22.41        2.51  -0.13  34.80      96.20
ivol          100000 2000.00     11.91        2.92  -0.06  14.96      94.92
                      off        19.42        2.87  -0.06  25.32      93.70
fixed 1%/fill 10000  2000.00      0.47       -1.13  -0.69  59.24      93.33
equal         10000  2000.00      2.61        2.63  -0.35  60.39      85.42
                      off         3.18        2.84  -0.36  59.02      85.50
ivol          10000  2000.00      1.58        1.88  -0.31  56.15      73.57
                      off         1.76        2.29  -0.29  54.82      73.47
fixed 1%/fill 100000 2000.00      1.10        1.29  -0.10  13.45      99.44
equal         100000 2000.00      1.14        1.61  -0.10  14.26     100.00
                      off         1.29        1.92  -0.13  18.17     100.00
ivol          100000 2000.00      1.06        0.91  -0.10  13.29      93.53
                      off         1.10        1.02  -0.14  17.07      93.10
fixed 1%/fill 10000  2000.00      0.49       -1.07  -0.65  59.55      94.01
equal         10000  2000.00      1.55        1.70  -0.39  65.65      84.57
                      off         2.41        2.56  -0.36  59.31      88.72
ivol          10000  2000.00      1.60        2.01  -0.30  51.69      78.99
                      off         1.70        2.21  -0.29  52.12      78.26
fixed 1%/fill 100000 2000.00      1.11        1.50  -0.10  11.26      99.50
equal         100000 2000.00      1.14        1.86  -0.10  11.63     100.00
                      off         1.30        2.23  -0.13  13.04     100.00
ivol          100000 2000.00      1.06        1.09  -0.10  10.78      95.81
                      off         1.11        1.21  -0.14  12.57      94.59

### Continuous-book equity curve and monthly multiples

One continuous run over the whole period (train + both test windows) for the chosen
configuration: equal mimicry, $10k book, capped.

In [27]:
import plotly.graph_objects as go

res_all = simulate(d[d['day'] >= TRAIN_START], bhat, B0=10_000.0, return_eq=True)
eq_all = res_all['eq']

fig = go.Figure()
fig.add_trace(go.Scatter(x=eq_all.index, y=eq_all.values, mode='lines', name='equity'))
fig.add_shape(type='line', x0=str(TRAIN_END.date()), x1=str(TRAIN_END.date()),
              y0=0, y1=1, yref='paper', line=dict(color='red', dash='dash'))
fig.add_annotation(x=str(TRAIN_END.date()), y=1, yref='paper',
                   text='train end', showarrow=False, yanchor='bottom')
fig.update_yaxes(type='log')
fig.update_layout(
    width=1100, height=420,
    title=f'copy book equity, equal mimicry $10k -> ${eq_all.iloc[-1]:,.0f} '
          f'({res_all["multiple"]:.1f}x), maxDD {res_all["maxDD"]:+.0%}')
fig.show()

print('monthly multiples:')
print((eq_all.resample('ME').last().pct_change() + 1).round(3).to_string())

monthly multiples:
2025-09-30 00:00:00+00:00    NaN
2025-10-31 00:00:00+00:00   2.82
2025-11-30 00:00:00+00:00   0.94
2025-12-31 00:00:00+00:00   1.32
2026-01-31 00:00:00+00:00   1.54
2026-02-28 00:00:00+00:00   3.66
2026-03-31 00:00:00+00:00   1.72
2026-04-30 00:00:00+00:00   2.11
2026-05-31 00:00:00+00:00   1.05
2026-06-30 00:00:00+00:00   1.41
2026-07-31 00:00:00+00:00   1.00
2026-08-31 00:00:00+00:00   1.01
2026-09-30 00:00:00+00:00   1.00
Freq: ME


### Random-wallet placebo: is it the selection, or just being active?

Sample `PLACEBO_SIM_N` sets of 53 random wallets from the full politics universe
(>= 100 train fills each), run the identical pipeline (B-hat from their own train fills,
equal mimicry, same limits), compare multiples per split.

In [28]:
pool_cnt = df_full[df_full['is_train']].groupby('wallet', observed=True)['notional'].size()
pool = sorted(set(pool_cnt[pool_cnt >= 100].index) - set(copyable_wallets))
print(f'placebo pool: {len(pool):,} wallets (>=100 train fills)')

rng = np.random.default_rng(42)
pl_rows = []
for b in range(PLACEBO_SIM_N):
    ws = set(rng.choice(pool, size=N, replace=False))
    subp = df_full[df_full['wallet'].isin(ws)].copy()
    subp['res_ts'] = pd.to_datetime(subp['last_condition_trade_ts'], utc=True)
    day_p = subp['dt'].dt.floor('D')
    trw = subp[subp['is_train'] & (day_p >= TRAIN_START)]
    bh_p = (trw.groupby(['wallet', 'condition_id'], observed=True)['notional'].sum()
               .groupby(level=0).quantile(0.75))
    row = {'set': b}
    row['train'] = simulate(trw, bh_p)['multiple']
    row['test_c'] = simulate(subp[(~subp['is_train']) & (day_p >= TRAIN_START)], bh_p)['multiple']
    row['test_t'] = simulate(subp[day_p > TRAIN_END], bh_p)['multiple']
    pl_rows.append(row)
    print(f"placebo {b}: train={row['train']:7.2f}  test_c={row['test_c']:6.2f}  "
          f"test_t={row['test_t']:6.2f}")

pl_sim = pd.DataFrame(pl_rows).set_index('set')
act_mults = {s_name: simulate(sub_act, bhat)['multiple']
             for s_name, sub_act in SIM_SPLITS.items()}
summ = pd.DataFrame({
    'actual': act_mults,
    'placebo median': pl_sim.median(),
    'placebo p10': pl_sim.quantile(0.1),
    'placebo p90': pl_sim.quantile(0.9),
})
summ['P(actual<=placebo)'] = [(pl_sim[s] >= act_mults[s]).mean() for s in summ.index]
summ

placebo pool: 4,375 wallets (>=100 train fills)


placebo 0: train=   3.28  test_c=  1.80  test_t=  1.80


placebo 1: train=   4.25  test_c=  0.72  test_t=  0.53


placebo 2: train=   9.28  test_c=  0.66  test_t=  0.14


placebo 3: train=   0.00  test_c=  0.93  test_t=  1.01


...

placebo 7: train=   6.02  test_c=  1.67  test_t=  1.71


placebo 8: train=   3.21  test_c=  0.63  test_t=  0.71


placebo 9: train=   0.02  test_c=  0.32  test_t=  0.48
placebo 10: train=   7.26  test_c=  0.49  test_t=  0.50


placebo 11: train=   0.00  test_c=  1.80  test_t=  1.69


,actual,placebo median,placebo p10,placebo p90,P(actual<=placebo)
train,114.19,3.24,0.00,9.08,0.00
test_c,2.61,0.71,0.50,1.78,0.00
test_t,1.55,0.74,0.48,1.71,0.25


### Replication findings

- **Mimicry beats naive fixed staking decisively out-of-sample**: at a $10k book,
  fixed 1%/fill *loses* money in both test windows while bankroll-proportional copying
  roughly doubles the bankroll. Preserving the wallets' own conviction sizing is what
  carries the edge - flat staking destroys it.
- **The selection edge survives trade-level simulation.** Actual multiples sit above all
  random-wallet placebo sets on the contract split and far above placebo medians on every
  split. Part of the absolute return is regime beta (random active wallets also made
  money); the alpha is the spread versus them.
- **Capacity, not methodology, is now the binding constraint.** At $10k the liquidity
  ceiling already binds on most fills yet returns stay strong; at $100k ~100% of fills
  hit the ceiling and per-window returns compress to +14%/+30%. The strategy scales down
  gracefully but has a hard capacity wall.
- **Equal vs inverse-vol weights: no consistent OOS winner** in the simulator either -
  consistent with everything above, keep it simple.
- Caveats: cash accounting without mark-to-market, `copyable_qty_20m_100` assumes fills
  within ~20 minutes of the trigger trade at its price, no fees/slippage modeling.
  Stage 2 execution-tape work would tighten all three.

## Conclusions (adversarial pass)

1. **The in-sample vol reduction is noise-mining.** The placebo permutation test shows that
   fitting minvar on *mutually independent* shuffled wallet series yields the same ~66%
   in-sample reduction as the real data; the actual value sits well inside the placebo
   distribution. 53 free parameters against a 303-day noisy covariance is enough to
   manufacture arbitrary-looking in-sample "diversification".
2. **There is little co-movement to harvest at any aggregation frequency.** Mean pairwise
   correlation of copyable(20m) daily pnl is ~+0.08 (hourly +0.02, weekly +0.15); the
   per-contract directional grid (~0.45-0.48) measures same-side betting, not pnl
   synchronization. Switching aggregation (days vs markets) does not create exploitable
   structure - it confirms its absence.
3. **OOS ranking inverts vs in-sample**: equal-scaled (worst in-sample) is the best OOS
   variant; minvar and even diagonal-only inverse-vol weights degrade OOS. More in-sample
   optimization pressure => worse out-of-sample, the classic signature of overfitting.
4. **Regime break**: universe copyable pnl collapses ~98% right after TRAIN_END
   (Jun 2026 ~1.0M/month -> Jul/Aug 2026 ~16k/2k), so any train-fitted portfolio faces a
   structurally different OOS regime regardless of weighting.
5. **Per-contract notional caps genuinely help diversification and make the covariance
   signal real** (placebo percentile jumps from ~30th to ~100th at every cap 1k-5k), but
   they do not make the minvar combination persist forward (block folds still fail) and
   they truncate the win tail the universe's pnl depends on.
6. **Mechanism v2 works**: bankroll-mimicking proportional replication with liquidity
   ceilings turns the selected wallets into an investable book whose edge survives OOS
   windows and random-wallet placebos - while naive fixed staking loses money OOS. The
   wallet selection carries real information about *sizing*, not just direction. Capacity,
   not methodology, is the binding constraint (~100% of fills liquidity-bound at a $100k
   book).
7. **Clustering answers the small-sample objection**: aggregating wallets to K=8 groups
   makes the in-sample vol reduction statistically real (above all shuffle placebos, vs
   ~30th percentile at full dimensionality), so estimation noise did hide part of the
   covariance signal at 53 assets. But detecting the signal is not exploiting it: OOS and
   WFCV vol reduction stay negative at every K and shrinkage level because
   cross-correlation structure rotates across regimes and cluster membership is unstable
   over time (split-half ARI ~ 0).

Implication: a static covariance-minimizing combination of these wallets does not produce
a stable lower-vol copyable stream - drop minvar entirely. The defensible setup is:
per-contract notional caps for risk control + bankroll-proportional replication of the
selected wallets at a book size where the liquidity ceiling binds on a minority of fills.
If pursued further, candidates are quarterly refits evaluated only within-regime,
selecting wallets on *timing* similarity (daily-cos grid), or stage 2 execution modeling
(slippage/fees/mark-to-market) to pin down realistic capacity.